# 02. Realistic Long-Short Quantile Backtest
Backtest a simple strategy with fees (0.04% maker/taker avg) and funding rates.

In [ ]:
import sys
sys.path.append('..')
from loader import load_panel
import polars as pl

# Strategy: Long bottom 20% funding (receivers), Short top 20% funding (payers)
# 1h rebalancing

df = load_panel("2022-01-01", "2022-02-01").collect()

# Calculate signal
signal = (
    df.with_columns([
        pl.col("next_funding_rate").rank().over("timestamp").alias("rank") / pl.col("symbol").count().over("timestamp")
    ])
    .with_columns([
        pl.when(pl.col("rank") > 0.8).then(-1)
          .when(pl.col("rank") < 0.2).then(1)
          .otherwise(0).alias("position")
    ])
)

# Calculate returns with costs
COST_BPS = 0.0004
pnl = (
    signal.sort(["symbol", "timestamp"])
    .with_columns([
        (pl.col("position").shift(1) * pl.col("ret_1h")).alias("gross_ret"),
        (pl.col("position").diff().abs() * COST_BPS).alias("txn_cost"),
        (pl.col("position") * pl.col("funding_rate")).alias("funding_cost") # simplified
    ])
    .with_columns([
        (pl.col("gross_ret") - pl.col("txn_cost") - pl.col("funding_cost")).alias("net_ret")
    ])
    .group_by("timestamp").agg(pl.col("net_ret").mean())
    .sort("timestamp")
    .with_columns(pl.col("net_ret").cum_sum().alias("equity"))
)

print(pnl.tail())